<a href="https://colab.research.google.com/github/JMK9904/ba_framing_yt-kommentare/blob/main/BA_Datenaufbereitung.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bertopic sentence-transformers umap-learn hdbscan gensim optuna nltk dask[complete]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.2 MB/s eta 0:00:00


In [ ]:
import optuna
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
import umap
import hdbscan
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from pathlib import Path
import pandas as pd
import numpy as np
import re
import dask.dataframe as dd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
comments_all = dd.read_parquet("/content/drive/MyDrive/Bachelor Thesis/public_media_data/full_data.parquet")

In [ ]:
print(comments_all['source_file'].value_counts().compute())
print(comments_all['source_file'].value_counts(normalize=True).compute().mul(100).round(1).astype(str) + '%')

source_file
yt_comments_tagesschau_2024_Q1     62109
yt_comments_tagesschau_2024_Q4     24228
yt_comments_tagesschau_2025_Q1     59100
yt_comments_tagesschau_2025_Q4     25084
yt_comments_zdfheute_2023_Q3      260756
yt_comments_tagesschau_2023_Q3     20849
yt_comments_tagesschau_2023_Q4     40418
yt_comments_tagesschau_2024_Q2     50159
yt_comments_tagesschau_2025_Q3     25385
yt_comments_zdfheute_2024_Q2      395281
yt_comments_zdfheute_2025_Q2      248074
yt_comments_tagesschau_2023_Q2     28785
yt_comments_tagesschau_2025_Q2     36930
yt_comments_zdfheute_2023_Q4      450133
yt_comments_zdfheute_2024_Q3      275566
yt_comments_zdfheute_2024_Q4      347250
yt_comments_zdfheute_2025_Q1      511881
yt_comments_zdfheute_2025_Q4      109166
yt_comments_tagesschau_2024_Q3     27074
yt_comments_zdfheute_2023_Q2      159552
yt_comments_zdfheute_2024_Q1      505731
yt_comments_zdfheute_2025_Q3      251017
Name: count, dtype: int64[pyarrow]
source_file
yt_comments_tagesschau_2024_Q1     1.6%

In [ ]:
ddf = comments_all

In [ ]:
ddf["channel_year_q"] = ddf["source_file"].str.replace(
    r"^yt_comments_",
    "",
    regex=True
)

ddf[["channel", "year", "quarter"]] = (
    ddf["channel_year_q"]
    .str.rsplit("_", n=2, expand=True)
)

ddf = ddf.drop(columns="channel_year_q")
print(type(ddf))
#ddf["channel"] = ddf.categorize(columns=["channel", "quarter"])
#ddf["year"] = ddf["year"].astype("int16")

<class 'dask.dataframe.dask_expr._collection.DataFrame'>


In [ ]:
print(ddf["channel"].value_counts().compute())
print(ddf["channel"].value_counts(normalize=True).compute().mul(100).round(1).astype(str) + '%')

channel
tagesschau     400121
zdfheute      3514407
Name: count, dtype: int64[pyarrow]
channel
tagesschau    10.2%
zdfheute      89.8%
Name: proportion, dtype: object


# NEUE FILTERMETHODE

In [ ]:
import re

def load_regex_dictionary(path):
    with open(path, "r", encoding="ISO-8859-1") as f:
        patterns = [
            line.strip()
            for line in f
            if line.strip() and not line.startswith("#")
        ]
    return patterns

regex_terms = load_regex_dictionary("/content/drive/MyDrive/Bachelor Thesis/soldisk_wörterbuch.txt")

combined_pattern = re.compile(
    "(" + "|".join(regex_terms) + ")",
    flags=re.IGNORECASE
)


In [ ]:
with open("/content/drive/MyDrive/Bachelor Thesis/soldisk_wörterbuch.txt", encoding="ISO-8859-1") as f:
    patterns = [line.strip() for line in f if line.strip()]
combined_pattern_str = "|".join(patterns)

In [ ]:
print(type(combined_pattern))
print(combined_pattern)

<class 're.Pattern'>
re.compile('([Rr]assis.*|Vaterland.*|.*[Aa]syl.*|Refugee.*|[Dd]isplaced|.*[Ss]chutzberechtigt.*|.*[Ss]chutzsuchend.*|Resettlement.*|[Ss]ubsidiär.*|.*[Mm]igra.*|migrier.*|[Ee]inwander.*|[Ee]ingewandert.*|[Zz]uwan, re.IGNORECASE)


In [ ]:
with open("/content/drive/MyDrive/Bachelor Thesis/soldisk_wörterbuch.txt", encoding="ISO-8859-1") as f:
    terms = [line.strip() for line in f if line.strip()]

combined_pattern = "|".join(terms)

print(type(combined_pattern))
print(combined_pattern)

<class 'str'>
[Rr]assis.*|Vaterland.*|.*[Aa]syl.*|Refugee.*|[Dd]isplaced|.*[Ss]chutzberechtigt.*|.*[Ss]chutzsuchend.*|Resettlement.*|[Ss]ubsidiär.*|.*[Mm]igra.*|migrier.*|[Ee]inwander.*|[Ee]ingewandert.*|[Zz]uwander.*|[Zz]ugewandert.*|[Aa]uswander.*|[Aa]usgewandert.*|Ausländer.*|.*[Aa]ussied.*|Sudetendeutsche.*|Volksdeutsche.*|.*[Vv]ertriebene.*|Zwangsarbeit.*|Gastarbeit.*|Fremdarbeit.*|Saisonarbeit.*|[Rr]usslanddeutsch.*|Familienzusammenführung.*|[Uu]nbegleitete.*|.*[Ss]taatsangehörig.*|.*[Ss]taatenlos.*|Einbürgerung.*|Bleiberecht.*|[Ii]ntegration.*|.*[Ff]reizügigkeit.*|.*[Aa]bschiebung.*|[Ii]slam.*|[Mm]uslim.*|[Jj]esid.*|Schengen.*|Dublin.*|UNHCR.*|UNRRA.*|UNRWA.*|IRO.*|IOM.*|.*[Ff]lüchtl.*|[Hh]erkunft.*|[Mm]uttersprach.*|.*[Ff]l[uü]cht.*|[Ff]liehen.*|.*flohen.*|Landarbeit.*|Anwerbe.*|Aufenthalts.*|.*[Aa]ssimil.*|.*[Aa]ufnahme.*|.*[Rr]ückführung.*|.*[Aa]usweisung.*


In [ ]:
#from nltk.tokenize.casual import regex
'''
filtered_ddf = ddf[
    ddf["text"].str.contains(
        combined_pattern,
        case=False,
        na=False,
        regex=True
    )
]

filtered_df = filtered_ddf.compute()'''


filtered_ddf = ddf[
    ddf["text"].str.contains(
        combined_pattern,
        case=False,
        na=False,
        regex=True
    )
]

filtered_df = filtered_ddf.compute()

In [ ]:
print(len(filtered_ddf))
print(filtered_ddf["channel"].value_counts().compute())
print(filtered_ddf["channel"].value_counts(normalize=True).compute().mul(100).round(1).astype(str) + '%')

240423
channel
tagesschau     22891
zdfheute      217532
Name: count, dtype: int64[pyarrow]
channel
tagesschau     9.5%
zdfheute      90.5%
Name: proportion, dtype: object


In [ ]:
###########################################
###########################################
###########hat es funktioniert?############
###########################################
###########################################

In [ ]:
len_before = len(comments_all)
len_after = filtered_ddf.shape[0].compute()

print(len_before, "→", len_after)
print("Retained:", len_after / len_before)

3914528 → 240423
Retained: 0.06141813265865003


In [ ]:
filtered_ddf["text"].sample(frac=.01).compute()

,text
323425,"In der Frage, ob wir Flüchtlinge kreppieren la..."
363536,Die ganze Runde hat nicht die größte Bedrohung...
197344,Wir brauchen die 180-Wende in der Migrationspo...
390120,Die russischen Bots sagen was von Frauen schla...
809023,Das Problem ist das sind keine Fachk...
...,...
3021354,Islam verbieten
3700204,Vor einigen Jahren hat das ZDF noch die Berich...
2969063,"""Brain-Drain"" wird dass auch genannt. Und es i..."
3505433,Vollkommen richtig! Es interessiert die Welt n...


In [ ]:
non_filtered = ddf[~ddf["text"].str.contains(combined_pattern, case=False, regex=True)]
non_filtered["text"].sample(frac=0.01).compute()

,text
249177,du bvergisst das viele auch als Terroristen ab...
524940,Es ist erstaunlich: Der nette Herr von der CDU...
835588,"Herr Lanz, warum befragen Sie einen Grünen üb..."
788964,"@illuminatusprimus3683 Ich bin gegen Merz, abe..."
963584,ZDF muss auf den freien Markt 🤣🤣🤣🙈
...,...
3012905,das ist Richtig. Israel ist ein demokratische...
3642355,"Es gibt genug Leute, die Habeck für einen fähi..."
3014766,"Tja, diese Welt macht es einem nicht leicht mi..."
3722806,Netanjahu und Putin sind Kriegsverbrecher. PUN...


In [ ]:
import re

pattern = re.compile(combined_pattern, re.IGNORECASE)

def extract_match(text):
    m = pattern.search(text)
    return m.group(0) if m else None

sample = filtered_ddf.sample(frac=0.01).compute()
sample["match"] = sample["text"].apply(extract_match)

sample["match"].value_counts().head(20)

,count
match,
Islam.,5
auswandern,3
Islam,3
Ironie off,3
Rassisten.,3
Islamisten!,2
auswandern.,2
"Rassist, ein wahnsinnig schlechter Unternehmer (er hat 400 Millionen vom Vater bekommen und es auch irgendwie geschafft mit Casinos pleite zu gehen), gibt ein F für Frauen(-Rechte), verherrlicht Diktatoren wie Hitler, unterwirft sich Diktatoren wie Putin und Kim Jong-Un, scherrt sich null um andere demokratische Länder (würde Ukraine instantly aufgeben für ""Frieden""), hat die gute Wirtschaft Obamas (er war nicht gut mit der Wirtschaft) zerstört/zunichte gemacht, hatte eine sehr hohe Arbeitslosigkeitsrate, hat den Angriff auf das US Kapitol angezettelt, hat Corona verhamlost und so 1000-10000 Menschen sinnlos ""geopfert"", hat den Reichen Steuererniedrigungen gegeben und den Mittelstand weiter belastest, will noch auf jedes Land 10% (auf China 60%) Zölle erheben was Deutschland über 4 Jahre mehr als 150 Milliarden kosten wird, aus der Nato aussteigen, verharmlost den Klimawandel komplett und der erste Präsident der je verurteilt wurde. Nun nenne mir mal mindestens die gleiche Menge an Sachen die man auf dem gleichen Level bei Harris kritisieren könnte?! Ich warte...",2
Integrationskurse angesagt!,2


In [ ]:
zdf = filtered_ddf[filtered_ddf["channel"] == "zdfheute"]
tagesschau = filtered_ddf[filtered_ddf["channel"] == "tagesschau"]

zdf_sampled = zdf.sample(frac=0.105, random_state=42)

balanced = dd.concat([zdf_sampled, tagesschau])

In [ ]:
print(len(balanced))
print(balanced["channel"].value_counts().compute())
print(balanced["channel"].value_counts(normalize=True).compute().mul(100).round(1).astype(str) + '%')

45733
channel
zdfheute      22842
tagesschau    22891
Name: count, dtype: int64[pyarrow]
channel
zdfheute      49.9%
tagesschau    50.1%
Name: proportion, dtype: object


In [ ]:
comments_soldisk = balanced

In [ ]:
comments_soldisk.head()

,video_id,video_title,video_published,comment_id,parent_id,likes,is_reply,text,comment_published,author,source_file,channel,year,quarter
345290,HKdwe64Cu6o,Wie Amerika Jagd auf Migranten macht | ausland...,2025-07-18T17:25:31Z,UgyrYGIz2oJL8eJYmSF4AaABAg,<NA>,2,False,Für die Migranten tut es mir sehr leid. Eine r...,2025-07-20T11:50:53Z,@Samuraischwert,yt_comments_zdfheute_2025_Q3,zdfheute,2025,Q3
485179,WFGtcJ2pbZM,Kabinett bringt Wehrdienst-Gesetz auf den Weg:...,2025-08-27T12:01:05Z,Ugz0Ze4FmOZhqjq82TR4AaABAg.AML02yr0z-GAML4pkj2AL8,Ugz0Ze4FmOZhqjq82TR4AaABAg,0,True,Es ist viel schlimmer. Illegale Migranten müss...,2025-08-27T18:24:04Z,@johannjohannes8265,yt_comments_zdfheute_2025_Q3,zdfheute,2025,Q3
397643,N21qQUVMnIk,heute 19:00 Uhr vom 04.08.2025 Streit um Bürge...,2025-08-04T18:48:54Z,UgwjvpSJz5owN-0nCJd4AaABAg,<NA>,3,False,Am besten finde ich die Aussage der gegenseite...,2025-08-04T19:38:18Z,@SH-vq6uj,yt_comments_zdfheute_2025_Q3,zdfheute,2025,Q3
347125,yshd_WVeyXI,heute journal vom 18.07.2025 Pressekonferenz v...,2025-07-18T22:20:00Z,UgyIdTSgzjKq-sX51Kl4AaABAg,<NA>,2,False,Die Aufnahmezusagen für die Äthiopier hat Baer...,2025-07-19T07:57:59Z,@pablobreitner8626,yt_comments_zdfheute_2025_Q3,zdfheute,2025,Q3
333970,80xVMNtEGo8,heute 19 Uhr vom 15.07.25 Brosius-Gersdorf Sta...,2025-07-15T19:00:30Z,Ugy0ZcIswG9GRWarNwF4AaABAg.AKcK5tPZV2SAKcQSLH3BQj,Ugy0ZcIswG9GRWarNwF4AaABAg,4,True,@MrBillirock möge der Himmel heute über Sie BL...,2025-07-16T04:16:30Z,@zotemicha4156,yt_comments_zdfheute_2025_Q3,zdfheute,2025,Q3


In [ ]:
comments_soldisk.to_parquet("/content/drive/MyDrive/Bachelor Thesis/public_media_data/filtered_data.parquet")

In [ ]:
ddf.to_parquet("/content/drive/MyDrive/Bachelor Thesis/public_media_data/full_data.parquet")

#Deskriptive Maße

In [ ]:
comments_soldisk["char_count"] = comments_soldisk["text"].str.len()
comments_soldisk["word_count"] = comments_soldisk["text"].str.split().str.len()

comments_soldisk[["char_count", "word_count"]].describe()

,char_count,word_count
count,30611.000000,30611.000000
mean,315.447584,47.315148
std,495.798297,73.293814
min,4.000000,1.000000
25%,95.000000,14.000000
50%,179.000000,27.000000
75%,348.000000,53.000000
max,9701.000000,1494.000000
